# Práctica de la clase 8: programa tu propio SVM lineal

En esta práctica retomas la clase SVM (máquina de vectores de soporte, del inglés *Support Vector Machine*) que construiste en clase. El método `fit`, el que busca los valores de `w` y `b` probando y descartando candidatos, ya está resuelto: tu trabajo es completar dos piezas que quedaron pendientes.

Encuentras dos ejercicios. En el primero completas el método `predict`. En el segundo completas el código que dibuja el hiperplano y los márgenes con Matplotlib. Cada ejercicio trae un enunciado con pistas, una celda de código con partes marcadas como `# TODO` para que completes, y más abajo una solución colapsada por si te atascas. Intenta resolverlo por tu cuenta antes de mirarla.

Trabajas sobre un conjunto de puntos distinto al de clase, generado también con `make_blobs`. No necesitas ninguna conexión a internet ni ninguna API externa.

## Ejercicio 1: completa el método predict

Tienes preparado el conjunto de puntos y una clase SVM con el método `fit` ya resuelto, exactamente como lo programaste en clase. Falta el método `predict`, que debe aplicar la fórmula `signo(w . x + b)` y devolver la clase (`1` o `-1`) del punto que le pases.

**Pistas:**

- Convierte `features` en un array de NumPy con `np.array()` antes de operar con `self.w`
- Usa `np.dot()` para el producto escalar entre el vector de pesos y el punto, y súmale `self.b`
- `np.sign()` te devuelve directamente el signo del resultado: `1.0`, `-1.0` o `0.0`
- Si la visualización está activa (`self.visualizacion`) y la clasificación no es cero, dibuja el punto como una estrella con `self.ax.scatter(features[0], features[1], s=200, marker='*', c=self.colores[clasificacion])`, igual que hiciste en clase
- Si algo falla al ejecutar la celda de verificación, revisa primero que tu método termine con `return clasificacion`; sin ese `return`, la función devuelve `None` y las comparaciones no tienen sentido

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=10, centers=2, n_features=2, cluster_std=0.8,
                   center_box=(2, 10), random_state=21)

# La clase SVM espera un diccionario {-1: puntos, 1: puntos}
datos = {
    -1: X[y == 0],
    1: X[y == 1]
}

print("Puntos de la clase -1:")
print(datos[-1])
print("\nPuntos de la clase 1:")
print(datos[1])

In [ ]:
class SVM:
    def __init__(self, visualizacion=True):
        self.visualizacion = visualizacion
        self.colores = {1: 'r', -1: 'b'}
        if self.visualizacion:
            self.fig = plt.figure()
            self.ax = self.fig.add_subplot(1, 1, 1)
        self.w = None
        self.b = None

    def fit(self, data):
        self.data = data
        opt_dict = {}
        transformaciones = [[1, 1], [-1, 1], [-1, -1], [1, -1]]

        todos_los_valores = []
        for clase in self.data:
            for punto in self.data[clase]:
                for valor in punto:
                    todos_los_valores.append(valor)

        self.max_valor = max(todos_los_valores)
        self.min_valor = min(todos_los_valores)
        todos_los_valores = None

        pasos = [self.max_valor * 0.1,
                 self.max_valor * 0.01,
                 self.max_valor * 0.001]

        b_rango_multiplo = 5
        b_paso_multiplo = 5
        ultimo_optimo = self.max_valor * 10

        for paso in pasos:
            w = np.array([ultimo_optimo, ultimo_optimo])
            optimizado = False

            while not optimizado:
                for b in np.arange(-1 * self.max_valor * b_rango_multiplo,
                                    self.max_valor * b_rango_multiplo,
                                    paso * b_paso_multiplo):
                    for transformacion in transformaciones:
                        w_candidato = w * transformacion
                        cumple_restriccion = True

                        for clase in self.data:
                            for punto in self.data[clase]:
                                yi = clase
                                if not yi * (np.dot(w_candidato, punto) + b) >= 1:
                                    cumple_restriccion = False
                                    break
                            if not cumple_restriccion:
                                break

                        if cumple_restriccion:
                            opt_dict[np.linalg.norm(w_candidato)] = [w_candidato, b]

                if w[0] < 0:
                    optimizado = True
                else:
                    w = w - paso

            normas = sorted([n for n in opt_dict])
            eleccion = opt_dict[normas[0]]
            self.w = eleccion[0]
            self.b = eleccion[1]
            ultimo_optimo = eleccion[0][0] + paso * 2

    def predict(self, features):
        # TODO 1: calcula la clasificación aplicando signo(w . x + b)
        #         con np.sign, np.dot, np.array, self.w y self.b
        clasificacion = ...

        # TODO 2: si la visualización está activa y la clasificación no es
        #         cero, dibuja el punto como una estrella (marker='*', s=200)
        #         del color que corresponda a su clase (self.colores)
        ...

        return clasificacion

Ejecuta la siguiente celda de verificación. `fit` no usa descenso de gradiente, así que puede tardar unos segundos en encontrar `w` y `b`. Si tu `predict` está bien completado, los puntos cercanos al grupo etiquetado como `-1` deberían clasificarse como `-1`, y los cercanos al grupo etiquetado como `1`, como `1`.

In [ ]:
svm = SVM()
svm.fit(datos)

print("w =", svm.w)
print("b =", svm.b)

puntos_a_predecir = [[2, 5], [3, 3], [8, 2], [7, 1.5], [5, 3.5]]

for punto in puntos_a_predecir:
    print(punto, svm.predict(punto))

<details>
<summary>💡 Ver solución</summary>

```python
    def predict(self, features):
        # TODO 1: signo(w . x + b): el producto escalar del punto, convertido
        # en array de NumPy, por el vector de pesos, más el sesgo
        clasificacion = np.sign(np.dot(np.array(features), self.w) + self.b)

        # TODO 2: si la visualización está activa y el punto no cae justo
        # sobre el hiperplano, lo dibujamos como una estrella de su color
        if clasificacion != 0 and self.visualizacion:
            self.ax.scatter(features[0], features[1], s=200, marker='*',
                             c=self.colores[clasificacion])

        return clasificacion
```

Con `predict` completo, el modelo clasifica cada punto nuevo según el lado del hiperplano en el que cae: los más cercanos al grupo `-1` obtienen esa clase, y los más cercanos al grupo `1`, la clase `1`. El quinto punto, `[5, 3.5]`, queda a medio camino entre ambos grupos y aun así el modelo lo asigna a un lado concreto, el que le corresponde según la posición exacta del hiperplano.

</details>

## Ejercicio 2: completa la visualización del hiperplano y los márgenes

Ahora completas `visualizar`, el método que dibuja los puntos de entrenamiento, el hiperplano de decisión y sus dos márgenes. Esta vez partes de una clase SVM con `fit` y `predict` ya resueltos, igual que los programaste en clase, y de un conjunto de puntos ya entrenado.

**Pistas:**

- Para dibujar los puntos de entrenamiento, recorre `self.data` (un diccionario `{-1: puntos, 1: puntos}`) y usa `self.ax.scatter(punto[0], punto[1], s=100, color=self.colores[clase])` para cada punto de cada clase
- La función auxiliar `hiperplano(x, w, b, desplazamiento)` despeja la `y` de la ecuación `w.x + b = desplazamiento`; la fórmula es `(-w[0] * x - b + desplazamiento) / w[1]`
- `x_min` y `x_max` ya están calculados; evalúa `hiperplano` en ambos para obtener los dos extremos de cada línea, y dibújala con `self.ax.plot([x_min, x_max], [y1, y2], estilo)`
- Los tres valores de `desplazamiento` que necesitas son `1` (margen positivo), `-1` (margen negativo) y `0` (hiperplano de decisión); dibuja los dos márgenes con estilo `'k'` y el hiperplano con estilo `'y--'`
- Termina con `plt.show()` para que el gráfico se muestre

In [ ]:
class SVM:
    def __init__(self, visualizacion=True):
        self.visualizacion = visualizacion
        self.colores = {1: 'r', -1: 'b'}
        if self.visualizacion:
            self.fig = plt.figure()
            self.ax = self.fig.add_subplot(1, 1, 1)
        self.w = None
        self.b = None

    def fit(self, data):
        self.data = data
        opt_dict = {}
        transformaciones = [[1, 1], [-1, 1], [-1, -1], [1, -1]]

        todos_los_valores = []
        for clase in self.data:
            for punto in self.data[clase]:
                for valor in punto:
                    todos_los_valores.append(valor)

        self.max_valor = max(todos_los_valores)
        self.min_valor = min(todos_los_valores)
        todos_los_valores = None

        pasos = [self.max_valor * 0.1,
                 self.max_valor * 0.01,
                 self.max_valor * 0.001]

        b_rango_multiplo = 5
        b_paso_multiplo = 5
        ultimo_optimo = self.max_valor * 10

        for paso in pasos:
            w = np.array([ultimo_optimo, ultimo_optimo])
            optimizado = False

            while not optimizado:
                for b in np.arange(-1 * self.max_valor * b_rango_multiplo,
                                    self.max_valor * b_rango_multiplo,
                                    paso * b_paso_multiplo):
                    for transformacion in transformaciones:
                        w_candidato = w * transformacion
                        cumple_restriccion = True

                        for clase in self.data:
                            for punto in self.data[clase]:
                                yi = clase
                                if not yi * (np.dot(w_candidato, punto) + b) >= 1:
                                    cumple_restriccion = False
                                    break
                            if not cumple_restriccion:
                                break

                        if cumple_restriccion:
                            opt_dict[np.linalg.norm(w_candidato)] = [w_candidato, b]

                if w[0] < 0:
                    optimizado = True
                else:
                    w = w - paso

            normas = sorted([n for n in opt_dict])
            eleccion = opt_dict[normas[0]]
            self.w = eleccion[0]
            self.b = eleccion[1]
            ultimo_optimo = eleccion[0][0] + paso * 2

    def predict(self, features):
        clasificacion = np.sign(np.dot(np.array(features), self.w) + self.b)
        if clasificacion != 0 and self.visualizacion:
            self.ax.scatter(features[0], features[1], s=200, marker='*',
                             c=self.colores[clasificacion])
        return clasificacion

    def visualizar(self):
        # TODO 1: dibuja todos los puntos de entrenamiento (self.data),
        #         coloreados por clase, con self.ax.scatter
        ...

        def hiperplano(x, w, b, desplazamiento):
            # TODO 2: despeja la y de la ecuación w.x + b = desplazamiento
            ...

        rango_datos = (self.min_valor * 0.9, self.max_valor * 1.1)
        x_min = rango_datos[0]
        x_max = rango_datos[1]

        # TODO 3: calcula los dos puntos del margen positivo (desplazamiento=1)
        #         y dibuja la línea con self.ax.plot(..., 'k')
        ...

        # TODO 4: calcula los dos puntos del margen negativo (desplazamiento=-1)
        #         y dibuja la línea con self.ax.plot(..., 'k')
        ...

        # TODO 5: calcula los dos puntos del hiperplano de decisión
        #         (desplazamiento=0) y dibuja la línea con self.ax.plot(..., 'y--')
        ...

        plt.show()

Entrena este segundo modelo y llama a `visualizar` para comprobar tu trabajo. Deberías ver los puntos de entrenamiento de las dos clases, el hiperplano de decisión en amarillo discontinuo y los dos márgenes en negro continuo, uno a cada lado.

In [ ]:
svm2 = SVM()
svm2.fit(datos)

for punto in [[2, 5], [3, 3], [8, 2], [7, 1.5]]:
    svm2.predict(punto)

svm2.visualizar()

<details>
<summary>💡 Ver solución</summary>

```python
    def visualizar(self):
        # TODO 1: un punto por cada elemento de cada clase, coloreado según
        # self.colores
        [[self.ax.scatter(punto[0], punto[1], s=100, color=self.colores[clase])
          for punto in self.data[clase]] for clase in self.data]

        def hiperplano(x, w, b, desplazamiento):
            # TODO 2: despejamos la y de w.x + b = desplazamiento
            return (-w[0] * x - b + desplazamiento) / w[1]

        rango_datos = (self.min_valor * 0.9, self.max_valor * 1.1)
        x_min = rango_datos[0]
        x_max = rango_datos[1]

        # TODO 3: margen positivo, w.x + b = 1
        psv1 = hiperplano(x_min, self.w, self.b, 1)
        psv2 = hiperplano(x_max, self.w, self.b, 1)
        self.ax.plot([x_min, x_max], [psv1, psv2], 'k')

        # TODO 4: margen negativo, w.x + b = -1
        nsv1 = hiperplano(x_min, self.w, self.b, -1)
        nsv2 = hiperplano(x_max, self.w, self.b, -1)
        self.ax.plot([x_min, x_max], [nsv1, nsv2], 'k')

        # TODO 5: hiperplano de decisión, w.x + b = 0
        db1 = hiperplano(x_min, self.w, self.b, 0)
        db2 = hiperplano(x_max, self.w, self.b, 0)
        self.ax.plot([x_min, x_max], [db1, db2], 'y--')

        plt.show()
```

Las tres líneas son paralelas porque las tres se calculan con el mismo vector `w`, solo cambia el `desplazamiento` con el que despejas la `y`. El hiperplano de decisión queda siempre justo a medio camino entre los dos márgenes, a la misma distancia de cada uno.

</details>

## Resumen: lo que te llevas de esta práctica

- `predict` aplica `signo(w . x + b)` con `np.sign`, `np.dot` y `np.array`, y solo necesita dibujar el punto como estrella si la visualización está activa y la clasificación no es cero
- `visualizar` dibuja tres líneas paralelas despejando la `y` de `w.x + b = desplazamiento`, con `desplazamiento` igual a `1`, `-1` o `0`
- Las mismas dos piezas, `predict` y `visualizar`, funcionan igual de bien sobre un conjunto de puntos distinto al de clase, porque no dependen de los datos concretos, solo de `w`, `b` y `self.data`
- Comprobar el modelo con puntos de prueba situados claramente cerca de cada grupo es la forma más rápida de detectar un error antes de mirar el gráfico

**Siglas usadas en esta práctica:** SVM (Support Vector Machine, máquina de vectores de soporte).